# Model 1: Local Inference (No API)

**Goal:** Run all 644 Austrian tax law questions through a pre-trained German language model (dbmdz/german-gpt2) and save the answers.

This is the simplest model — we use a pre-trained model directly via HuggingFace. The model has never seen Austrian tax law specifically, so we expect low accuracy.

In [24]:
# Install required packages
!pip install transformers torch pandas

In [25]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load pre-trained German GPT-2 model
model_name = "dbmdz/german-gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set pad token
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Model: {model_name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {device}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: dbmdz/german-gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: dbmdz/german-gpt2
Parameters: 124,445,952
Device: cpu


In [ ]:
# Load the dataset
df = pd.read_csv("../dataset_clean.csv")
print(f"Loaded {len(df)} questions")
df.head(3)

In [33]:
# Generation function: format as Q&A prompt, let GPT-2 complete
def generate_answer(question, max_new_tokens=150):
    prompt = f"Frage: {question}\nAntwort:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_new_tokens=30,
            no_repeat_ngram_size=3,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the generated part (not the prompt)
    generated_tokens = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # Cut at the first "Frage:" if the model starts generating a new Q&A pair
    if "Frage:" in answer:
        answer = answer[:answer.index("Frage:")].strip()

    return answer

# Quick test
test_q = df.iloc[0]["prompt"]
print(f"Q: {test_q}")
print(f"A: {generate_answer(test_q)}")

Q: Was ist die steuerliche Bemessungsgrundlage für die Körperschaftsteuer?
A: Die!"#$%&'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLM


In [28]:
# Run inference on all questions
answers = []

for i, row in df.iterrows():
    answer = generate_answer(row["prompt"])
    answers.append(answer)
    if (i + 1) % 25 == 0 or i == 0:
        print(f"{i+1}/{len(df)} | {row['id']} | {answer[:80]}...")

1/643 | CORP-TAX-001 | Die...
25/643 | CORP-TAX-025 | Die...
50/643 | CORP-TAX-050 | Die...
75/643 | TAX-INTL-015 | Die...
100/643 | TAX-INTL-040 | Die...
125/643 | TAX-INTL-065 | Die...
150/643 | ESTG-NEW-010 | Herr...
175/643 | VAT-DOM-015 | Die...
200/643 | VAT-DOM-040 | Die...
225/643 | VAT-DOM-065 | Die...
250/643 | GRESt-AT-010 | Die...
275/643 | GRESt-AT-035 | Die...
300/643 | GRESt-AT-060 | Die...
325/643 | VAT-INTL-004 | "...
350/643 | VAT-INTL-029 | "...
375/643 | VAT-INTL-054 | "...
400/643 | VAT-INTL-079 | "...
425/643 | ESTG27-024 | Die...
450/643 | EMP-009 | Die...
475/643 | EMP-034 | Die...
500/643 | EMP-059 | Die...
525/643 | EStG-23-024 | Der...
550/643 | EStG-23-049 | Die...
575/643 | SELF-014 | Die...
600/643 | SELF-039 | Die...
625/643 | SELF-064 | Die...


In [29]:
# Save results as CSV
results = pd.DataFrame({"id": df["id"], "answer": answers})
results.to_csv("../results/model1_results.csv", index=False)
print(f"Saved {len(results)} answers to model1_results.csv")
results.head(3)

Saved 643 answers to model1_results.csv


,id,answer
0,CORP-TAX-001,Die
1,CORP-TAX-002,Die
2,CORP-TAX-003,Die
